# 4 · From curves to a table for statistics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansgomez87/py-mir-toolbox/blob/main/docs/chapters/04_statistics_export.ipynb)

*Before this chapter: [1 · Frames and dynamics](01_dynamics.ipynb), [2 · The spectrum and its shape](02_spectral_shape.ipynb), [3 · MFCCs and spectral flux](03_mfcc_flux.ipynb).*

The earlier chapters computed feature curves, one value per frame. This chapter prepares them for statistics:

- **summary statistics** (`mir.stat`) turn each curve into a few numbers per piece;
- `mir.batch` computes features for every audio file of a folder, and `mir.file_table` and `mir.to_csv` save them as a table with one row per file, ready for R, SPSS or JASP.

The chapter has the same steps as before: ① the idea; ② the maths in NumPy; ③ mirpy; ④ librosa; ⑤ Essentia; ⑥ the three libraries side by side; and an exercise.

## Setup

In Colab, the next cell installs the libraries. On your own computer, see *Getting started*.

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic(
        "pip", "install -q 'py-mir-toolbox==0.1.*' librosa essentia"
    )

In [ ]:
import tempfile
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio

import mirpy as mir

try:
    import essentia
    import essentia.standard as es

    essentia.log.infoActive = False  # hide Essentia's progress messages
except ImportError:
    es = None
    print("Essentia is not installed here; the Essentia cells will be skipped.")

sr = 22050  # sampling rate of the clips saved below

## Summary statistics and a table per folder

### ① Idea

A feature curve has one value per frame: thousands of values for a song. For statistics, such as comparing pieces or relating features to listeners' ratings, we usually want **one row per piece** with a few numbers per feature. MIRtoolbox's `mirstat` gives these summaries:

- **Mean** and **Std** (standard deviation): the average level and how much it varies;
- **Slope**: whether the feature tends to rise or fall over the excerpt;
- **PeriodFreq**, **PeriodAmp** and **PeriodEntropy**: whether the curve repeats regularly, how fast, and how clearly.

`mirstat` leaves out frames without a value (NaN). We use four freely licensed recordings from librosa's examples, 30 seconds each:

In [ ]:
clips = {
    "brahms": "Brahms, Hungarian Dance No. 5; US Army Strings (public domain)",
    "nutcracker": "Tchaikovsky, Dance of the Sugar Plum Fairy; arr. Kevin MacLeod (CC BY 4.0)",
    "trumpet": "Mihai Sorohan, solo trumpet 06 (CC BY 4.0)",
    "vibeace": "Kevin MacLeod, Vibe Ace (CC BY 4.0)",
}
for name, credit in clips.items():
    print(f"{name:>10}: {credit}")

In [ ]:
trumpet = mir.audio(librosa.ex("trumpet"))
Audio(trumpet.data[0, :, 0], rate=trumpet.sr)

### ② From scratch

Mean, standard deviation and slope of the trumpet's centroid curve. For the slope, MIRtoolbox first rescales the time axis of all frames to run from −0.5 to 0.5, drops the frames without a value, rescales the curve (mean 0, standard deviation 1), and fits a straight line through 0.

In [ ]:
centroid = mir.centroid(trumpet, frame=True)
values = centroid.data[0, 0]
keep = ~np.isnan(values)  # frames with a value
times = centroid.frame_centers[keep]
curve = values[keep]

scaled_time = np.linspace(-0.5, 0.5, values.size)[keep]
scaled_curve = (curve - curve.mean()) / curve.std(ddof=1)
slope = np.linalg.lstsq(scaled_time[:, np.newaxis], scaled_curve, rcond=None)[0][0]

print(f"mean {curve.mean():.0f} Hz, std {curve.std(ddof=1):.0f} Hz, slope {slope:.2f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times, curve, color="gray")
ax.axhline(curve.mean(), color="C0", label="mean")
ax.axhspan(
    curve.mean() - curve.std(ddof=1), curve.mean() + curve.std(ddof=1), alpha=0.15, label="± std"
)
ax.plot(times, curve.mean() + slope * scaled_time * curve.std(ddof=1), color="C3", label="slope")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Centroid (Hz)")
ax.legend()
plt.show()

### ③ mirpy

`mir.stat` is MIRtoolbox `mirstat`. It returns a dictionary with one entry per summary.

In [ ]:
summary = mir.stat(centroid)
for field, value in summary.items():
    print(f"{field:>13}: {value.data.item():.3f}")

`PeriodFreq` and `PeriodAmp` are NaN here: the trumpet's centroid does not repeat at a regular rate.

To do the same for a whole folder, `mir.batch` runs a set of features on every audio file. It replaces MIRtoolbox's `'Folder'` keyword. By default, each framed feature is summarised by its mean and standard deviation, as `mirexport` does. `mir.file_table` turns the result into a table with one row per file, and `mir.to_csv` saves it.

First we save the four clips as WAV files in a temporary folder:

In [ ]:
folder = Path(tempfile.mkdtemp())
for name in clips:
    y, clip_sr = librosa.load(librosa.ex(name), sr=sr, duration=30)
    sf.write(folder / f"{name}.wav", y, clip_sr)
print(sorted(path.name for path in folder.glob("*.wav")))

In [ ]:
features = {
    "rms": lambda x: mir.rms(x, frame=True),
    "lowenergy": mir.lowenergy,
    "centroid": lambda x: mir.centroid(x, frame=True),
    "flux": mir.flux,
}
rows = mir.batch(folder, features)
table = mir.file_table(rows)
mir.to_csv(table, folder / "features.csv")

df = pd.read_csv(folder / "features.csv")
df

This CSV file opens directly in R (`read.csv`), SPSS or JASP. A chart of two columns:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
clip_names = df["file"].str.removesuffix(".wav")
axes[0].bar(clip_names, df["RMS_Mean"])
axes[0].set_title("Average RMS")
axes[1].bar(clip_names, df["Spectralcentroid_Mean"])
axes[1].set_title("Average spectral centroid (Hz)")
plt.show()

For the curves of one file, frame by frame, use `summary=False` and `mir.frame_table`. All features in a frame table must share the same frames.

In [ ]:
trumpet_frames = mir.frame_table(
    {
        "rms": mir.rms(trumpet, frame=True),
        "centroid": mir.centroid(trumpet, frame=True),
    }
)
pd.DataFrame(trumpet_frames).head()

### ④ librosa

librosa has no batch function or summaries. A loop over the files and a pandas `DataFrame` do the job:

In [ ]:
librosa_rows = []
for path in sorted(folder.glob("*.wav")):
    y, _ = librosa.load(path, sr=None)
    librosa_rows.append(
        {
            "file": path.name,
            "rms_mean": np.mean(librosa.feature.rms(y=y)),
            "centroid_mean": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
        }
    )
librosa_df = pd.DataFrame(librosa_rows)
librosa_df

### ⑤ Essentia

Essentia's `MusicExtractor` computes a large set of features, with their statistics, in one call per file. Its results are named like `lowlevel.spectral_centroid.mean`. It uses its own settings (for example, longer frames), not MIRtoolbox's.

In [ ]:
if es is None:
    print("Essentia is not available; skipped.")
else:
    extractor = es.MusicExtractor(lowlevelStats=["mean", "stdev"])
    essentia_rows = []
    for path in sorted(folder.glob("*.wav")):
        pool = extractor(str(path))[0]
        essentia_rows.append(
            {
                "file": path.name,
                "centroid_mean": pool["lowlevel.spectral_centroid.mean"],
            }
        )
    essentia_df = pd.DataFrame(essentia_rows)
    print(essentia_df)

### ⑥ Side by side

The average centroid of each clip from the three libraries:

In [ ]:
comparison = pd.DataFrame(
    {
        "mirpy": df["Spectralcentroid_Mean"].to_numpy(),
        "librosa": librosa_df["centroid_mean"].to_numpy(),
    },
    index=df["file"],
)
if es is not None:
    comparison["Essentia"] = essentia_df["centroid_mean"].to_numpy()

comparison.plot.bar(figsize=(8, 3), rot=0)
plt.ylabel("Average centroid (Hz)")
plt.show()

- All three find Vibe Ace and the Nutcracker the darkest clips.
- The trumpet and Brahms swap places. The trumpet clip has many near-silent frames between notes. mirpy leaves them out (as MIRtoolbox does), while librosa and Essentia average them in, and the faint noise in those frames has a high centroid.
- The averages also differ because each library uses its own frame length and spectrum settings.
- **mirpy** gives MIRtoolbox's summaries and column names, so existing MATLAB analyses can be redone. **librosa** leaves the summaries to you. **Essentia**'s `MusicExtractor` is the quickest way to get many features at once.

### Exercise

Add the brightness and the MFCCs to the batch, write the table to a CSV file, and read it back with pandas. Which clip has the highest average brightness? How many columns do the MFCCs add?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
more_features = {
    "brightness": lambda x: mir.brightness(x, frame=True),
    "mfcc": lambda x: mir.mfcc(x, frame=True),
}
more = pd.DataFrame(mir.file_table(mir.batch(folder, more_features)))
print(more[["file", "Brightness_Mean"]])
print(sum(column.startswith("MFCC") for column in more.columns), "MFCC columns")

Each of the 13 coefficients gets a mean and a standard deviation: 26 columns.

## Summary

| MIRtoolbox | mirpy | librosa | Essentia |
|---|---|---|---|
| `mirstat(f)` | `mir.stat(f)` | NumPy + pandas | `MusicExtractor` statistics |
| `mirrms('Folder')` + `mirexport` | `mir.batch` + `mir.file_table` + `mir.to_csv` | loop + pandas | `MusicExtractor` per file |

## Next

The next chapters, on pitch, tonality and rhythm, come with release 0.2.

## References

As cited in the MIRtoolbox 1.8.1 User's Manual.

- Lartillot, O., Toiviainen, P. "A Matlab Toolbox for Musical Feature Extraction From Audio", International Conference on Digital Audio Effects, Bordeaux, 2007. *MIRtoolbox.*